# EVE 310 - Lab 10: Clustering with k-means — activity

**Module 4 | 11/05/2026**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThyanRevolter/eve310-fall-2026/blob/main/labs/lab10-clustering/notebooks/lab10-activity.ipynb)

## Learning objectives

By the end of this lab you will be able to:

1. Standardize the columns of a dataset before clustering, and say why it matters
2. Choose the number of clusters from the elbow and silhouette curves
3. Turn centroids back into real units and describe what each cluster is

## Before you start

1. Click **Copy to Drive** at the top of this window and work in the copy that opens. Colab throws away anything you did not copy when the runtime ends.
2. Run the setup cell below before anything else. It downloads the two data files into `DATA_DIR` and creates `FIGURES_DIR`.
3. Work down the notebook in order. Replace every `--` with your code and keep the variable names exactly as given. Three short questions: standardize and fit, choose k, name the clusters.
4. Keep `random_state=0, n_init=10` in every `KMeans` call, as in the tutorial: it makes everyone's clusters identical, so the autograder and the quiz can refer to them.
5. When you are done: **Runtime > Restart session and run all**, then **File > Download > Download .ipynb**, and upload that file to Gradescope. The autograder gives you feedback on each question, but **this submission is not graded** and does not count toward your final grade. The Lab 10 quiz on Canvas is what counts, and it asks about what you compute here, so finish the activity before you take it.

## 0. Setup

In [ ]:
# EVE 310 setup - run this cell first, every time you open this notebook.
# It downloads this lab's data files into a "data" folder in your Colab session.
import pathlib
import urllib.request

LAB = "lab10-clustering"
DATA_FILES = ["lecture_points.csv", "household_water.csv"]

DATA_DIR = pathlib.Path("data")
FIGURES_DIR = pathlib.Path("figures")
DATA_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

BASE_URL = f"https://raw.githubusercontent.com/ThyanRevolter/eve310-fall-2026/main/labs/{LAB}/data"
for name in DATA_FILES:
    if not (DATA_DIR / name).exists():
        urllib.request.urlretrieve(f"{BASE_URL}/{name}", DATA_DIR / name)

print(f"Ready. {len(DATA_FILES)} data file(s) in {DATA_DIR.resolve()}")

**The data (run as is)**

A water utility installed smart meters on 300 households. For each household the file `household_water.csv` has two numbers: `gallons_per_day`, the average daily consumption, and `night_flow_gph`, the smallest flow recorded between 2 and 4 am, in gallons per hour. A household with no leak uses almost nothing at that hour, so a steady night flow is the classic sign of a leak: a running toilet, a dripping irrigation valve, a cracked service line. The utility wants to know what **kinds** of households it has, and in particular whether there is a group it should send a leak inspector to. Nobody has labeled a single household, so this is a clustering job.

Look at the ranges in the table below before you go on: `gallons_per_day` runs from about 120 to 660, `night_flow_gph` from 0 to 30.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

water_df = pd.read_csv(DATA_DIR / 'household_water.csv')
X = water_df[['gallons_per_day', 'night_flow_gph']].to_numpy()

print(len(water_df), 'households')
print(water_df.describe().round(1))

fig, ax = plt.subplots()
ax.scatter(X[:, 0], X[:, 1], s=20, color='k')
ax.set_xlabel('gallons per day')
ax.set_ylabel('night flow (gallons per hour)')
ax.set_title('300 households, no labels')

**k-means on the raw columns (run as is)**

First, the tutorial's recipe applied as is: `k = 3` on the two columns exactly as they come. Read the three centroids and the three cluster sizes, and look at where the colors change on the plot.

In [ ]:
kmeans_raw = KMeans(n_clusters=3, random_state=0, n_init=10).fit(X)
labels_raw = kmeans_raw.labels_

print('centroids (gallons per day, night flow):')
print(np.round(kmeans_raw.cluster_centers_, 1))
print('households per cluster:', np.bincount(labels_raw))

fig, ax = plt.subplots()
ax.scatter(X[:, 0], X[:, 1], c=labels_raw, s=20)
ax.scatter(kmeans_raw.cluster_centers_[:, 0], kmeans_raw.cluster_centers_[:, 1], marker='x', s=250, color='red', linewidths=3)
ax.set_xlabel('gallons per day')
ax.set_ylabel('night flow (gallons per hour)')
ax.set_title('k = 3 on the raw columns')

The colors change along the **horizontal** axis only: the clusters are "low, medium and high gallons per day", and the households with a night flow of 20 or 30 gallons per hour are scattered across two of them. That is not a bug in k-means. It measures the straight-line distance between households, and a difference of 100 gallons per day counts for exactly as much as a difference of 100 gallons per hour of night flow, which no household ever shows. On these units the night flow is invisible to the distance, so the clusters ignore it.

The fix is Lab 6's: **standardize** each column, so that both are in standard deviations from their own mean. Then one unit means the same thing in both directions.

## Your turn

**Question 1 — Standardize, then cluster**

Turn both columns into z-scores with the Lab 6 formula, (value - mean) / standard deviation, using the sample standard deviation (`ddof=1`). The means and standard deviations are computed for you, and you will need them again in Question 3. Use those four numbers, not `StandardScaler` and not `np.std` without `ddof=1`: those divide by the population standard deviation, the clusters come out the same, but the `Standardized?` line below prints False and the Question 3 checks fail. Then fit k-means with `k = 3` on the standardized columns.

- `X_scaled`: the two standardized columns as a NumPy array, one row per household
- `labels`: the cluster of every household (from the model fitted on `X_scaled`)
- `centroids_scaled`: the three centroids, in standardized units
- `sizes`: how many households are in each cluster

The check lines confirm that each standardized column has mean 0 and standard deviation 1, and that the labels and the centroids come from one three-cluster fit of `X_scaled`.

In [ ]:
gallons_mean = np.mean(water_df['gallons_per_day'])
gallons_std = np.std(water_df['gallons_per_day'], ddof=1)
night_mean = np.mean(water_df['night_flow_gph'])
night_std = np.std(water_df['night_flow_gph'], ddof=1)

# Each column as a z-score: (value - mean) / standard deviation
water_df['gallons_z'] = --
water_df['night_z'] = --
X_scaled = water_df[['gallons_z', 'night_z']].to_numpy()

# k-means with 3 clusters on the standardized columns (keep random_state=0, n_init=10)
kmeans = --
kmeans.fit(X_scaled)
labels = --
centroids_scaled = --
sizes = np.bincount(labels)

print('centroids (standardized):')
print(np.round(centroids_scaled, 2))
print('households per cluster:', sizes)
print('Standardized?', np.allclose(np.mean(X_scaled, axis=0), 0) and np.allclose(np.std(X_scaled, axis=0, ddof=1), 1))
print('Three clusters from one fit, every household in one of them?',
      len(centroids_scaled) == 3 and sizes.sum() == len(water_df)
      and np.allclose(centroids_scaled, [X_scaled[labels == c].mean(axis=0) for c in range(3)]))

The three cluster sizes barely moved, but the clusters are not the same households. Look at the standardized centroids in the printout: one of them sits about 2 standard deviations from the mean in one direction. Which direction (gallons per day, or night flow) sets that cluster apart from the other two, and why could the raw run not find it?

*Your answer:*

**Question 2 — Choose k**

k = 3 was a guess. Run the tutorial's two loops in one: for each k from 2 to 8, fit a model on `X_scaled` and store both its SSE and the silhouette score of its labels. Then take the k with the largest silhouette score and plot the two curves side by side.

- `sse_array`, `sc_array`: the SSE and the silhouette score for each k in `k_array`
- `best_k`: the k with the largest silhouette score
- `fig_k`: the two curves, SSE against k on the left axes and the silhouette score against k on the right

The check line confirms that the SSE falls at every step, that the k = 3 pass of the loop reproduces Question 1's model (same data, same `random_state`), and that `best_k` is one of the values in `k_array`.

In [ ]:
k_array = np.arange(2, 9)
sse_array = np.zeros(len(k_array))
sc_array = np.zeros(len(k_array))

for i in range(len(k_array)):
    # One model per pass, on X_scaled, with the current k
    kmeans_k = --
    sse_array[i] = --
    sc_array[i] = --

best_k = --

print('SSE:       ', np.round(sse_array, 1))
print('silhouette:', np.round(sc_array, 3))
print('best k by silhouette:', best_k)
print('SSE falls at every step, the k = 3 pass matches Question 1, and best_k is in the grid?',
      bool(np.all(np.diff(sse_array) < 0)) and np.isclose(sse_array[1], kmeans.inertia_)
      and sc_array.max() > sc_array.min() and best_k in k_array)

fig_k, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
# Two panels: ax1 is the left one, ax2 the right one
# SSE against k on ax1, the silhouette score against k on ax2, each with 'o-' (circles joined by lines) as in the tutorial
--
--
ax1.set_xlabel('k')
ax1.set_ylabel('SSE')
ax1.set_title('Elbow curve')
ax2.set_xlabel('k')
ax2.set_ylabel('silhouette score')
ax2.set_title('Silhouette curve')
fig_k.savefig(FIGURES_DIR / 'lab10_choose_k.png', dpi=150)

Do the elbow and the silhouette agree here? The SSE keeps falling past `best_k`; say in one sentence why that is not a reason to pick a larger k.

*Your answer:*

**Question 3 — Name the clusters**

A centroid in standardized units ("1.49 standard deviations above the mean") means little to the utility. Undo the z-score for each column of `centroids_scaled`: value = z × standard deviation + mean, with the means and standard deviations from Question 1 (Lab 6 did this to get back to kWh). Then find the cluster whose centroid has the highest night flow: that is the group the leak inspector visits.

- `centroids_gallons`: the three centroids' `gallons_per_day`, in gallons per day
- `centroids_night`: the three centroids' `night_flow_gph`, in gallons per hour
- `leak_cluster`: the cluster number whose centroid has the highest night flow
- `n_leak`: how many households are in that cluster
- `fig_clusters`: the households in their original units, colored by `labels`, with the three centroids drawn as red crosses

The first check line uses a fact about k-means: a centroid is the mean of its cluster's points, so a centroid converted back to real units must equal the mean of the households with that label, column by column. The second confirms that `leak_cluster` and `n_leak` agree with your own centroids and labels. The table `cluster_df` is for you to read.

In [ ]:
# Back to the original units: z * std + mean, one column at a time
centroids_gallons = --
centroids_night = --

cluster_df = pd.DataFrame({'cluster': np.arange(3), 'n_households': sizes,
                           'gallons_per_day': centroids_gallons, 'night_flow_gph': centroids_night})

# The cluster whose centroid has the highest night flow, and how many households it holds
leak_cluster = --
n_leak = --

print(cluster_df.round(1))
print('Leak cluster:', leak_cluster, 'with', n_leak, 'households')
print('Centroids match the cluster means, column by column?',
      np.allclose(centroids_gallons, [np.mean(water_df['gallons_per_day'][labels == c]) for c in range(3)])
      and np.allclose(centroids_night, [np.mean(water_df['night_flow_gph'][labels == c]) for c in range(3)]))
print('leak_cluster has the highest night flow, and n_leak counts its households?',
      centroids_night[leak_cluster] == np.max(centroids_night) and n_leak == np.sum(labels == leak_cluster))

fig_clusters, ax = plt.subplots()
# Every household in gallons per day and night flow, colored by its label
--
ax.scatter(centroids_gallons, centroids_night, marker='x', s=250, color='red', linewidths=3)
ax.set_xlabel('gallons per day')
ax.set_ylabel('night flow (gallons per hour)')
ax.set_title('k = 3 on the standardized columns')
fig_clusters.savefig(FIGURES_DIR / 'lab10_clusters.png', dpi=150)

Describe the three clusters in words a utility manager would use (one phrase each, from `cluster_df`). Then: a new household reports 310 gallons per day and a night flow of 20 gallons per hour. Before you can ask `kmeans.predict` where it belongs, what has to happen to those two numbers, and with which means and standard deviations?

*Your answer:*

**Quiz scratch space**

The Canvas quiz asks about the values above. Use the cell below during the quiz.

**DO NOT OVERWRITE ANY VARIABLES ABOVE THIS CELL. CREATE NEW VARIABLES AS NEEDED.**

In [ ]:
# Use this cell for quiz calculations

## Wrap-up

**Runtime > Restart session and run all**, confirm every cell runs top to bottom, then **File > Download > Download .ipynb** and upload it to Gradescope for feedback. The submission is not graded. Keep this notebook open for the Canvas quiz.